## Create Schema and Volume

Before we start cleaning data, we need a separate place to keep everything organized. 

- **Schema** = A folder inside your catalog that holds tables
- **Volume** = A folder where you can store files (like our CSV)

In [0]:
%sql
-- Create a new schema
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.ecommerce_cleaning;

-- Create a new volume
CREATE VOLUME IF NOT EXISTS cyntexa_dev.ecommerce_cleaning.raw_data;

## Load Dataset into DataFrame

A **DataFrame** is like a smart Excel sheet that lives inside Spark. We read the CSV file and turn it into a DataFrame so we can clean it using code.


In [0]:
# Read the CSV file from the volume into a DataFrame
df = spark.read\
    .option("header" , "true")\
    .option("inferSchema" , "true")\
    .csv("/Volumes/cyntexa_dev/ecommerce_cleaning/raw_data/Ecommerce/")

# Show the first 10 rows so we can see what the data looks like
display(df.limit(10))

In [0]:
# Show the structure (column names, data types, whether nulls are allowed)
df.printSchema()

## Find Columns with Null Values

**Null** means "no data" or "empty." We need to find which columns have missing values before we can clean them.

We use `.filter()` to find rows where a column is empty, and `.count()` to count them.

In [0]:
from pyspark.sql.functions import col, isnan, when, count , StringType

# Get the list of columns that are TEXT (StringType) — only these can be empty strings ""
string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType , StringType)]
# Count nulls and empty strings for each column
null_counts = df.select([
    count(
        when(
            # Condition 1: Value is NULL (works for ALL column types)
            col(c).isNull() |
            # Condition 2: OR value is empty string "" (ONLY for text columns, otherwise False)
            (col(c) == "" if c in string_columns else False),
            c
        )
    ).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# -----------------------------------------------
# Filter to see actual rows with nulls in specific columns
# -----------------------------------------------

# Show rows where 'Product Name' is null or empty
print("=== Rows with NULL or EMPTY Product Name ===")
display(df.filter((col("Product Name").isNull() | (col("Product Name") == ""))))

In [0]:
# Show rows where 'Category' is null or empty
print("=== Rows with NULL or EMPTY Category ===")
display(df.filter((col("Category").isNull() | (col("Category") == ""))))

In [0]:
# Show rows where 'Total Amount' is null
print("=== Rows with NULL Total Amount ===")
display(df.filter((col("Total Amount").isNull())))

In [0]:
# Show rows where 'Quantity' is null
print("=== Rows with NULL Quantity ===")
display(df.filter((col("Quantity").isNull())))

## Find Duplicate Rows

**Duplicate rows** are rows that are exactly the same as another row. We use `.dropDuplicates()` to find how many unique rows we have, then subtract that from the total to find duplicates.

We also use `.distinct()` to see unique combinations of data.

In [0]:
# Total number of rows
total_rows = df.count()
print(f"Total rows in dataset: {total_rows}")

# Number of unique rows (duplicates removed)
unique_rows = df.dropDuplicates().count()
print(f"Unique rows (no duplicates): {unique_rows}")

# Number of duplicate rows
duplicate_rows = total_rows - unique_rows
print(f"Duplicate rows: {duplicate_rows}")

# Find rows that appear more than once
from pyspark.sql.functions import count as spark_count

# Group by ALL columns and count how many times each combination appears
duplicates = df.groupBy(df.columns).agg(spark_count("*").alias("row_count"))\
                    .filter(col("row_count") > 1)
print("=== Rows that appear more than once ===")
display(duplicates)

In [0]:
# -----------------------------------------------
# Use distinct() to see unique values in a specific column
# -----------------------------------------------

print("=== Unique Product Names (before cleaning) ===")
display(df.select("Product Name").distinct())

In [0]:
print("=== Unique Categories (before cleaning) ===")
display(df.select("Category").distinct())


## Rename Columns to Snake Case

**Snake case** means writing names in lowercase with underscores instead of spaces.
For example:
- `Product Name` → `product_name`
- `Order ID` → `order_id`
- `Customer Name` → `customer_name`

We use `withColumnRenamed()` to rename columns one by one.

In [0]:
# Rename columns to snake_case using withColumnRenamed
# Syntax: withColumnRenamed("old_name", "new_name")

df_renamed = df\
    .withColumnRenamed("Order ID" , "order_id") \
    .withColumnRenamed("Product Name", "product_name") \
    .withColumnRenamed("Category", "category") \
    .withColumnRenamed("Customer Name", "customer_name") \
    .withColumnRenamed("Country", "country") \
    .withColumnRenamed("Order Date", "order_date") \
    .withColumnRenamed("Quantity", "quantity") \
    .withColumnRenamed("Unit Price", "unit_price") \
    .withColumnRenamed("Total Amount", "total_amount") \
    .withColumnRenamed("Payment Method", "payment_method") \
    .withColumnRenamed("Discount Applied", "discount_applied") \
    .withColumnRenamed("Shipping Status", "shipping_status")

# Show the new column names
print("New column names:")
print(df_renamed.columns)

# Show a sample of the renamed data
display(df_renamed.limit(5))

## Remove Duplicate Rows

After finding duplicates, we remove them using `.dropDuplicates()`. This keeps only one copy of each unique row and deletes the extras.

In [0]:
# Before removing duplicates
print(f"Rows before removing duplicates: {df_renamed.count()}")

# Remove duplicate rows
df_clean = df_renamed.dropDuplicates()

# After removing duplicates
print(f"Rows after removing duplicates: {df_clean.count()}")

# Show the cleaned data
display(df_clean.limit(10))

## Cleaning Pipeline - Intermediate Task

In this step, we build a complete data cleaning pipeline. Unlike basic tasks, here we make **judgment calls** about what "sensible defaults" mean for each column.

### Pipeline Steps:
1. **Drop fully-null rows** — Rows where EVERY column is empty. These add no value.
2. **Fill remaining nulls** — Replace empty values with logical defaults (not random!).
3. **Remove exact duplicates** — Keep only unique rows.
4. **Sort by order_date** — Arrange data chronologically.

### Our Judgment Calls (Sensible Defaults):
| Column | Default Value | Why? |
|--------|--------------|------|
| `product_name` | `"Unknown Product"` | We don't know what was ordered |
| `category` | `"Uncategorized"` | Can't guess the category |
| `customer_name` | `"Anonymous"` | Customer name missing |
| `country` | `"Unknown"` | Can't determine country |
| `quantity` | `1` | Most orders have at least 1 item |
| `total_amount` | `quantity × unit_price` | Calculate if possible, else `0` |
| `payment_method` | `"Not Specified"` | No payment info available |
| `discount_applied` | `"No"` | Assume no discount if not mentioned |
| `shipping_status` | `"Pending"` | Order hasn't been processed yet |

In [0]:
from pyspark.sql.functions import col , isnan , lit , coalesce , round , when
from pyspark.sql.types import StringType , IntegerType , DoubleType , DateType

# ============================================================
# STEP 0: Start with your DataFrame (use renamed one from previous step)
# ============================================================

df_pipeline = df_renamed 
print(f"Orignal Rows: {df_pipeline.count()}")

# ============================================================
# STEP 1: Drop fully-null rows
# ============================================================
# A "fully-null row" means EVERY column is NULL (or empty string for text)
# We check: drop rows where ALL columns are null
df_step1 = df_pipeline.dropna(how="all")
print(f"After dropping fully-null rows : {df_step1.count()}")

# ============================================================
# STEP 2: Fill remaining nulls with sensible defaults
# ============================================================

# First, let's also treat empty strings "" as nulls for text columns
# Because empty string is basically "no data"

string_columns = [field.name for field in df_step1.schema.fields if isinstance(field.dataType , StringType)]

# Replace empty strings with NULL, so we can fill them all together
for c in string_columns:
    df_step1 = df_step1.withColumn(
        c,
        when(col(c) == "",None).otherwise(col(c))
    )

# Now fill nulls with sensible defaults
df_step2 = df_step1.fillna({
    "product_name": "Unknown Product",
    "category": "Uncategorized",
    "customer_name": "Anonymous",
    "country": "Unknown",
    "quantity": 1,
    "payment_method": "Not Specified",
    "discount_applied": "No",
    "shipping_status": "Pending"
})

# For total_amount: if null, try to calculate quantity * unit_price
# If quantity is also null (now filled with 1), use that
df_step2 = df_step2.withColumn(
    "total_amount",
    when(
        col("total_amount").isNull(),
        round((col("quantity") * col("unit_price")) , 2)
    ).otherwise(col("total_amount"))
)
print("Sample after filling nulls:")
# display(df_step2.limit(5))

# ============================================================
# STEP 3: Remove exact duplicates
# ============================================================
df_step3 = df_step2.dropDuplicates()
print(f"After removing duplicates: {df_step3.count()}")
print(f"Duplicates removed: {df_step2.count() - df_step3.count()}")

# ============================================================
# STEP 4: Sort by order_date
# ============================================================
df_final = df_step3.orderBy(col("order_date").asc())
print("Final cleaned data (first 10 rows):")
display(df_final.limit(10))

# ============================================================
# STEP 5: Save the final cleaned table
# ============================================================
df_final.write\
        .mode("overwrite")\
        .saveAsTable("cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders")
print("✅ Pipeline complete! Table saved.")

In [0]:
%sql
select * from cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders 

## Read Orders and Regions Tables

We read both the cleaned orders table and the regions reference table into DataFrames so we can work with them.

In [0]:
# Read the cleaned orders table (from previous pipeline step)
df_orders = spark.table("cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders")

# Read the regions reference table from the volume
df_regions = spark.read\
                  .option("header" , "true")\
                  .option("inferSchema" , "true")\
                  .csv("/Volumes/cyntexa_dev/ecommerce_cleaning/raw_data/Reference Table/")

print("=== Orders Table ===")
df_orders.printSchema()
print(f"Orders row : {df_orders.count()}")

print("\n=== Regions Reference Table ===")
df_regions.printSchema()
print(f"Regions rows: {df_regions.count()}")

display(df_regions)

## Standardize Country Names Before Join

**Problem:** Our orders data has inconsistent country names like `"usa"`, `"USA"`, `"Usa"`. The reference table only has standard names like `"USA"`. If we join directly, many rows won't match!

**Solution:** Convert all country names to UPPERCASE so they match the reference table.

In [0]:
from pyspark.sql.functions import upper , trim , col

# Standardize: convert country to uppercase and remove extra spaces
df_orders_clean = df_orders.withColumn(
    "country",
    upper(trim(col("country")))
)

# Check unique countries after cleaning
print("Unique countries after standardizing:")
display(df_orders_clean.select("country").distinct())

# Also standardize the reference table just to be safe
df_regions_clean = df_regions.withColumn(
    "country_name",
    upper(trim(col("country_name")))
)

## Join Orders with Regions Reference Table

A **join** means combining two tables based on a common column. Here, both tables have a "country" column.

- **Left table:** `df_orders_clean` (main data)
- **Right table:** `df_regions_clean` (reference data)
- **Join type:** `LEFT JOIN` — keep ALL orders, even if country doesn't match any region
- **Join key:** `country` in orders = `country_name` in regions

In [0]:
from pyspark.sql.functions import col 

# LEFT JOIN: Keep all orders, add region info where country matches
df_joined = df_orders_clean.join(
    df_regions_clean,                       # Right table (reference)
    df_orders_clean.country == df_regions_clean.country_name,  # Join condition
    "left"                                   # Left join = keep all orders
)

# Show sample of joined data
print("Sample after join:")
display(df_joined.select( "order_id", "product_name", "country", "region", "country_code", "total_amount").limit(10))

In [0]:
# Check how many orders didn't match any region
unmatched = df_joined.filter(col("region").isNull()).count()
print(f"\nOrders with no matching region: {unmatched}")

## Aggregation 1: Revenue by Category

**Aggregation** = performing calculation after grouping data

Here we group by `category` and sum up the `total_amount` to see which product category earns the most revenue.

In [0]:
from pyspark.sql.functions import round , sum , count , col , countDistinct

# Group by category, calculate total revenue and number of orders
revenue_by_category = df_orders_clean.groupBy("category").agg(
    round(sum("total_amount"),2).alias("total_revenue"),  # Sum of all sales
    count("order_id").alias("total_orders") # Count of orders
).orderBy(col("total_revenue").desc()) # Highest revenue first

print("=== Revenue by Category ===")
display(revenue_by_category)

## Aggregation 2: Revenue by Region

Now that we've joined the regions table, we can group by `region` to see which part of the world generates the most revenue.

This is only possible because of the JOIN we did .

In [0]:
# Group by region (from the joined table), calculate revenue
revenue_by_region = df_joined.groupBy("region").agg(
    round(sum("total_amount") , 2).alias("total_revenue"),
    count("order_id").alias("total_orders"),
    countDistinct("country").alias("countries_count")  # How many countries in this region
).orderBy(col("total_revenue").desc())

print("=== Revenue by Region ===")
display(revenue_by_region)

## Save Aggregation Results

Save the aggregation results as tables so we can query them later with SQL or use them in dashboards.

In [0]:
# Save revenue by category
revenue_by_category.write\
                   .mode("overwrite")\
                   .saveAsTable("cyntexa_dev.ecommerce_cleaning.revenue_by_category")

# Save revenue by region
revenue_by_region.write\
                 .mode("overwrite")\
                 .saveAsTable("cyntexa_dev.ecommerce_cleaning.revenue_by_region")  

## Upload Dirty Dataset to Volume

We upload the advanced dirty dataset into our Databricks volume. This dataset contains real-world data quality problems that were not covered in the basic class exercises:

- **Currency symbols mixed with numbers** (e.g., `$49.99`, `€ 120`, `£1,299.50`)
- **Inconsistent date formats** (e.g., `15/03/2024`, `March 15, 2024`, `2024-03-15`)

## Read and Inspect Dirty Data

Before we clean anything, we must **understand the problems**. We load the data and look at the raw values to see exactly what kinds of dirt we are dealing with.

In [0]:
# Read the dirty CSV file
df_dirty = spark.read\
                .option("header" , "true")\
                .option("inferSchema" , "true")\
                .csv("/Volumes/cyntexa_dev/ecommerce_cleaning/raw_data/Dirty Ecommerce /")

print("=== Schema ===")
df_dirty.printSchema()

print("\n=== Total Rows ===")
print(df_dirty.count())

print("\n=== Sample of Dirty Prices ===")
display(df_dirty.select("dirty_price").distinct().limit(20))

print("\n=== Sample of Dirty Dates ===")
display(df_dirty.select("dirty_date").distinct().limit(20))

print("\n=== Null Counts ===")
from pyspark.sql.functions import col , isnan , when , count , StringType

# Step 1: Find which columns are TEXT (StringType)
string_column = [field.name for field in df_dirty.schema.fields if isinstance(field.dataType , StringType)]

null_counts = df_dirty.select([
    # Check 1: Is value NULL? (works for ALL columns)
    count(when(col(c).isNull() | 
    # Check 2: Is value empty string? (ONLY for text columns)          
    (col(c) == "" if c in string_column else False), c)).alias(c)
    for c in df_dirty.columns
])
display(null_counts)

## Clean Dirty Price Column

The `dirty_price` column contains prices in many different formats. Our goal is to:

1. **Extract the currency symbol** ($, €, £) from the text
2. **Remove symbols, spaces, and commas** to get a clean number
3. **Convert to a proper numeric type** (Double) so we can do math on it
4. **Store the original currency** so we can convert to a standard currency later

### Why this approach?
- We do **not** simply delete rows with symbols. That would lose real sales data.
- We use **regular expressions (regex)** to find patterns in text. Regex is the standard tool for parsing messy strings.

In [0]:
from pyspark.sql.functions import col , trim , when , isnan , lit , regexp_extract , regexp_replace

# Step 1: Extract the currency symbol using regex
# This looks for $, €, or £ at the start of the string
df_clean_price = df_dirty.withColumn(
    "currency_symbol",
    regexp_extract(col("dirty_price") , r"^[\s]*([\$\€\£])" , 1)
)

# display(df_clean_price)
# Step 2: Remove currency symbols, spaces, and commas from the price
# Keep only numbers and decimal points
df_clean_price = df_clean_price.withColumn(
    "price_numeric_string",
    regexp_replace(col("dirty_price") , r"[\$\€\£\s\,]" , "")
)

# Step 3: Convert the cleaned string to a Double (number)
# If the string is empty or invalid, it becomes NULL
df_clean_price = df_clean_price.withColumn(
    "price_numeric",
    col("price_numeric_string").cast("double")
)

display(df_clean_price)

In [0]:
# Step 4: Handle rows where the original price had no symbol (assume USD)
df_clean_price = df_clean_price.withColumn(
    "currency_symbol",
    when(col("currency_symbol") == "" , "USD")
    .when(col("currency_symbol") == "$", "USD")
    .when(col("currency_symbol") == "€", "EUR")
    .when(col("currency_symbol") == "£", "GBP")
    .otherwise("USD")  # Default fallback
)

# Show before and after
print("=== Before and After Cleaning ===")
display(df_clean_price.select(
    "dirty_price",
    "currency_symbol",
    "price_numeric_string",
    "price_numeric"
    ).limit(15))


## Clean Dirty Date Column

The `dirty_date` column contains dates in at least 6 different formats. We cannot simply use `to_date()` with one format because it would fail on most rows.

### Our Strategy: Try Multiple Formats
We attempt to parse the date using the most common formats, one by one. If the first format fails, we try the next. This is called **"coalesce parsing"** — we keep trying until one works.

### Formats We Handle
1. `yyyy-MM-dd` → `2024-03-15`
2. `dd/MM/yyyy` → `15/03/2024`
3. `MM-dd-yyyy` → `03-15-2024`
4. `MMMM dd, yyyy` → `March 15, 2024`
5. `dd-MMM-yyyy` → `15-Mar-2024`
6. `yyyy/MM/dd` → `2024/03/15`

### Why not just force one format?
If we forced everyone to use `yyyy-MM-dd`, we would **lose 80% of our data** because most rows would fail to parse. Real-world data is messy, and our job is to clean it, not throw it away.

In [0]:
from pyspark.sql.functions import try_to_date , coalesce

# Try to parse the date using multiple common formats
# coalesce picks the FIRST non-null result
df_standard = df_clean_price
df_dates = df_standard.withColumn(
    "order_date_clean",
    coalesce(
        try_to_date(col("dirty_date") , "yyyy-MM-dd"),
        try_to_date(col("dirty_date"), "dd/MM/yyyy"),
        try_to_date(col("dirty_date"), "MM-dd-yyyy"),
        try_to_date(col("dirty_date"), "MMMM dd, yyyy"),
        try_to_date(col("dirty_date"), "dd-MMM-yyyy"),
        try_to_date(col("dirty_date"), "yyyy/MM/dd")
    )
)

# Show before and after
print("=== Date Cleaning Results ===")
display(df_dates.select("dirty_date" , "order_date_clean").limit(20))

unparsed_dates = df_dates.filter(col("order_date_clean").isNull() & col("dirty_date").isNotNull()).count()
print(f"\nDates that could not be parsed: {unparsed_dates}")


## Create Final Clean Table

We select only the clean columns and give them proper names. The dirty columns are kept only for reference (so we can audit our work later), but we remove the intermediate working columns.

In [0]:
# Select only the clean, useful columns
df_final_advanced = df_dates.select(
    col("order_id"),
    col("product_name"),
    col("customer_country"),
    col("order_date_clean").alias("order_date"),
    col("quantity"),
    col("price_numeric").alias("price"),
    col("currency_symbol").alias("original_currency"),
    col("dirty_price").alias("original_price_raw"),  # Kept for audit
    col("dirty_date").alias("original_date_raw")      # Kept for audit
)

display(df_final_advanced.limit(10))

print("=== Final Clean Table Schema ===")
df_final_advanced.printSchema()

print(f"\nTotal clean rows: {df_final_advanced.count()}")

In [0]:
# Save the final clean table
df_final_advanced.write\
                 .mode("overwrite")\
                 .saveAsTable("cyntexa_dev.ecommerce_cleaning.advanced_cleaned_orders")

In [0]:
%sql
select * from cyntexa_dev.ecommerce_cleaning.advanced_cleaned_orders

## Task - 2 Cyntexa Analytics Repo — Branching Strategy & PR Workflow

This guide explains how our team manages code using branches and pull requests. Follow these steps to keep our work clean and safe.

---

### 1. Branching Strategy

We use two main branches:

| Branch | Purpose | Who works here |
|--------|---------|----------------|
| **`main`** | Production-ready code. This is the "source of truth." | Nobody works directly here |
| **`dev`** | Active development. All new features and fixes start here. | Everyone |

#### How it works:
- `main` always has stable, working code.
- `dev` is where we experiment, build, and test.
- When `dev` is stable, we merge it into `main` via a Pull Request.

#### Branch naming rules:
- Feature work: `feature/your-name-description`
  - Example: `feature/rahul-null-handling`
- Bug fixes: `fix/bug-description`
  - Example: `fix/date-format-error`

---

### 2. Daily Workflow for Teammates

Follow these steps every time you work on the project:

#### Step 1: Start from `dev`
Make sure you are on the `dev` branch before creating your own branch.

#### Step 2: Create your feature branch

#### Step 3: Do your work
Write code, test it, and commit regularly.

#### Step 4: Push your branch


---

### 3. Pull Request (PR) Review Workflow

Before any code goes into `main` or `dev`, it must be reviewed.

#### How to open a Pull Request:

1. Go to **GitHub** (or your Git provider).
2. Click **"Pull Requests"** → **"New Pull Request"**.
3. Set:
   - **Base branch:** `dev` (or `main` for releases)
   - **Compare branch:** your feature branch (`feature/rahul-null-handling`)
4. Write a clear title and description

## Load Clean Data for Analysis

We load the fully cleaned dataset into a Spark DataFrame and create a temporary view so we can run SQL queries on it.

### TASK 3
## Q1: Top Product-Country Combinations by Revenue

**Business Question:** Which product sells best in which country?

This helps the sales team focus marketing on high-performing product-country pairs.

In [0]:
%sql
SELECT 
product_name,
country,
ROUND(SUM(unit_price * quantity), 2) AS total_revenue,
COUNT(order_id) AS order_count
FROM cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders
GROUP BY product_name , country 
ORDER BY total_revenue DESC 
LIMIT 10;

## Q2: Monthly Revenue Trend

**Business Question:** How did our total revenue change month by month?

This shows whether the business is growing or slowing down over time.

In [0]:
%sql 
SELECT 
DATE_TRUNC('MONTH' , order_date) AS month,
ROUND(SUM(total_amount) , 2) AS total_revenue,
COUNT(order_id) AS order_count
FROM cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders
GROUP BY DATE_TRUNC('MONTH' , order_date)
ORDER BY month;

## Q3: Which Product Category Makes the Most Money?

**Business Question:** Which product category generated the highest total revenue?

This tells the business which product line to stock more of.

In [0]:
%sql 
SELECT category,
ROUND(SUM(total_amount) , 2) AS total_revenue,
COUNT(order_id) AS total_orders
FROM cyntexa_dev.ecommerce_cleaning.fully_cleaned_orders
GROUP BY category
ORDER BY total_revenue DESC;